# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

d

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
# Write your solution here

members_df = spark.sql("select * from members")
bookings_df = spark.sql("select * from bookings")
facilities_df = spark.sql("select * from facilities")

In [0]:
from pyspark.sql.functions import col, sum

result_df = bookings_df.filter(
    (col("starttime") >= "2012-09-01") &
    (col("starttime") < "2012-10-01")
).groupBy(
    "facid"
).agg(
    sum("slots").alias("total_slots")
).orderBy(
    "total_slots"
)

result_df.show()

+-----+-----------+
|facid|total_slots|
+-----+-----------+
|    5|        122|
|    3|        422|
|    7|        426|
|    8|        471|
|    6|        540|
|    2|        570|
|    1|        588|
|    0|        591|
|    4|        648|
+-----+-----------+



In [0]:
result_df.write.mode("overwrite").parquet("/FileStore/result_parquet")

## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
# Write your solution here

from pyspark.sql.functions import col, concat_ws

# Step 1: Build the result
result_df = bookings_df.alias("b") \
    .join(
        members_df.alias("m"),
        col("b.memid") == col("m.memid"),
        "inner"
    ) \
    .join(
        facilities_df.alias("f"),
        col("b.facid") == col("f.facid"),
        "inner"
    ) \
    .filter(col("f.name").like("Tennis Court%")) \
    .select(
        concat_ws(" ", col("m.firstname"), col("m.surname")).alias("member_name"),
        col("f.name").alias("facility")
    ) \
    .distinct() \
    .orderBy("member_name", "facility")

result_df.show(truncate=False)

# Step 2: Save as a Delta managed table partitioned by facility
result_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("facility") \
    .saveAsTable("threejoin_delta")

+--------------+--------------+
|member_name   |facility      |
+--------------+--------------+
|Anne Baker    |Tennis Court 1|
|Anne Baker    |Tennis Court 2|
|Burton Tracy  |Tennis Court 1|
|Burton Tracy  |Tennis Court 2|
|Charles Owen  |Tennis Court 1|
|Charles Owen  |Tennis Court 2|
|Darren Smith  |Tennis Court 2|
|David Farrell |Tennis Court 1|
|David Farrell |Tennis Court 2|
|David Jones   |Tennis Court 1|
|David Jones   |Tennis Court 2|
|David Pinker  |Tennis Court 1|
|Douglas Jones |Tennis Court 1|
|Erica Crumpet |Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|GUEST GUEST   |Tennis Court 1|
|GUEST GUEST   |Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows


## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
# Write your solution here

import os
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

headers = {
    "Content-Type": "application/json",
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": os.getenv("RAPIDAPI_KEY") or 'bcf6956c18msh65e79a4e25a794fp123224jsn4b976ce2d078'
}

symbols = {
    "Google": "GOOGL",
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Tesla": "TSLA"
}

rows = []

for company, symbol in symbols.items():
    params = {
        "function": "TIME_SERIES_DAILY",
        "symbol": symbol,
        "outputsize": "compact",
        "datatype": "json"
    }

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "Time Series (Daily)" not in data:
        print(f"API error for {company} ({symbol}): {data}")
        continue

    for dt, vals in data["Time Series (Daily)"].items():
        rows.append((
            company,
            symbol,
            dt,
            float(vals["1. open"]),
            float(vals["2. high"]),
            float(vals["3. low"]),
            float(vals["4. close"]),
            int(vals["5. volume"])
        ))

columns = ["company", "symbol", "date", "open", "high", "low", "close", "volume"]

stock_df = spark.createDataFrame(rows, columns)
stock_df = stock_df.withColumn("date", stock_df["date"].cast("date"))

stock_df.show(truncate=False)

+-------+------+----------+-------+--------+--------+------+--------+
|company|symbol|date      |open   |high    |low     |close |volume  |
+-------+------+----------+-------+--------+--------+------+--------+
|Google |GOOGL |2026-04-10|320.015|321.83  |316.32  |317.24|19152630|
|Google |GOOGL |2026-04-09|315.905|319.54  |311.06  |318.49|23739173|
|Google |GOOGL |2026-04-08|320.445|322.08  |315.02  |317.32|33547140|
|Google |GOOGL |2026-04-07|302.725|305.63  |297.72  |305.46|23205361|
|Google |GOOGL |2026-04-06|295.87 |300.62  |295.18  |299.99|16945494|
|Google |GOOGL |2026-04-02|290.69 |298.08  |289.45  |295.77|21666465|
|Google |GOOGL |2026-04-01|290.835|300.52  |290.41  |297.39|37684462|
|Google |GOOGL |2026-03-31|278.04 |288.08  |277.09  |287.56|43875400|
|Google |GOOGL |2026-03-30|276.42 |277.09  |272.11  |273.5 |35141244|
|Google |GOOGL |2026-03-27|277.275|279.37  |273.95  |274.34|35890612|
|Google |GOOGL |2026-03-26|287.91 |287.95  |278.5   |280.92|39080578|
|Google |GOOGL |2026

In [0]:
from pyspark.sql import functions as F

weekly_max_df = (
    stock_df
    .withColumn("year", F.year(F.col("date")))
    .withColumn("week", F.weekofyear(F.col("date")))
    .groupBy("company", "year", "week")
    .agg(F.max("close").alias("weekly_max_close"))
    .orderBy("company", "year", "week")
)

weekly_max_df.show(truncate=False)

+-------+----+----+----------------+
|company|year|week|weekly_max_close|
+-------+----+----+----------------+
|Apple  |2025|1   |273.76          |
|Apple  |2025|46  |272.41          |
|Apple  |2025|47  |271.49          |
|Apple  |2025|48  |278.85          |
|Apple  |2025|49  |286.19          |
|Apple  |2025|50  |278.78          |
|Apple  |2025|51  |274.61          |
|Apple  |2025|52  |273.81          |
|Apple  |2026|1   |271.01          |
|Apple  |2026|2   |267.26          |
|Apple  |2026|3   |261.05          |
|Apple  |2026|4   |248.35          |
|Apple  |2026|5   |259.48          |
|Apple  |2026|6   |278.12          |
|Apple  |2026|7   |275.5           |
|Apple  |2026|8   |264.58          |
|Apple  |2026|9   |274.23          |
|Apple  |2026|10  |264.72          |
|Apple  |2026|11  |260.83          |
|Apple  |2026|12  |254.23          |
+-------+----+----+----------------+
only showing top 20 rows


In [0]:
weekly_max_df.write \
    .mode("overwrite") \
    .partitionBy("company") \
    .saveAsTable("max_closing_price_weekly")

## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# Write your solution here

# Extract 100 RNA records from PostgreSQL using JDBC

jdbc_url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"

connection_properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

query = """
(
    SELECT *
    FROM rna
    LIMIT 100
) AS rna_subquery
"""

rna_df = spark.read.jdbc(
    url=jdbc_url,
    table=query,
    properties=connection_properties
)

rna_df.show(truncate=True)

+--------+-------------+--------------------+---------+----------------+---+--------------------+--------+--------------------+
|      id|          upi|           timestamp|userstamp|           crc64|len|           seq_short|seq_long|                 md5|
+--------+-------------+--------------------+---------+----------------+---+--------------------+--------+--------------------+
|41803978|URS00027DE0CA|2023-10-20 11:48:...|   rnacen|1AFE22AD960C571B| 98|ATCACCTGGAGATCTTG...|    NULL|363d833ba8aa41fa3...|
|41803979|URS00027DE0CB|2023-10-20 11:48:...|   rnacen|B12DC7B4BF59775A|143|TGTTAAAAATTTCATCT...|    NULL|363e3bff0e2fdcc49...|
|41803980|URS00027DE0CC|2023-10-20 11:48:...|   rnacen|40EF3B606E48BD5F| 67|TAAATTTTGGGACCGTC...|    NULL|363f5d9973b7da8ba...|
|41803981|URS00027DE0CD|2023-10-20 11:48:...|   rnacen|467F53E563BFC946|144|CACGGACAGGATTGACA...|    NULL|3640cacfd70890bcd...|
|41803982|URS00027DE0CE|2023-10-20 11:48:...|   rnacen|BAE2592489B49F74| 76|GAGACACCCTTCTGGGG...|    NUL

In [0]:
rna_df.write \
    .mode("overwrite") \
    .saveAsTable("rna_100_records")

In [0]:
spark.sql("SELECT * FROM rna_100_records").show(truncate=False)

+--------+-------------+--------------------------+---------+----------------+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------------------------------+
|id      |upi          |timestamp                 |userstamp|crc64           |len|seq_short                                                                                                                                                                              |seq_long|md5                             |
+--------+-------------+--------------------------+---------+----------------+---+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+--------------------------------+
|41803978|URS00027DE0CA|2023-10-20 11:48:14.848452|rnacen   |1AFE22AD960C